In [1]:
#-----------Neuro-symbolic graph reasoning---AI reads and understands the medical records, while predefined rules check whether the information is supported and consistent.
"""
Imagine a hospital has 500 pages of oncology records for one patient. You want to reconstruct the patient's treatment history.
There are three parts:
        1. Neural component — AI extracts information
            Reads clinical notes and identifies possible facts.
            Example: “Patient started osimertinib on 10 January 2026.”
            AI extracts: Drug = osimertinib; Start date = 10 January 2026.
        2. Symbolic component — rules verify the facts
            Checks whether the date, source and relationships satisfy predefined rules.
            Example: Does the record actually support that date? Is there another record giving a conflicting date?
        3. Graph component — connects the verified events
            Links diagnosis → biomarker test → treatment → response, while retaining links to the supporting records.
        
The combination is called neuro-symbolic graph reasoning: AI proposes facts, explicit rules check them, and a graph 
connects the resulting information.

--
For this demonstration, we'll simulate the output of an AI extraction system. The program then applies rules to decide 
which events can be accepted.

"""
# Simulated AI-extracted events from oncology documents. # A real system would use an NLP/LLM extraction pipeline.
events = [
    {
        "event" : "EGFR_Result",
        "date"  : "2026-01-10",
        "source": "Molecular_Report_12"
    },
    {
        "event" : "Treatment_Start",
        "date"  : "2026-01-20",
        "source": "Oncology_Note_15"
    },
    {
        "event" : "Treatment_Stop",
        "date"  : "2026-01-05",
        "source": "Oncology_Note_18"
    },
    {
        "event" : "Response_Assessment",
        "date"  : "2026-02-15",
        "source": None
    }
]
def validate_events(events):
    accepted = []
    flagged = []
    dates = {
        event["event"]: event["date"]
        for event in events
    }
    for event in events:
        problems = []
        if not event["source"]:                                             # Rule 1: Every event needs a supporting source.
            problems.append("Missing evidence source")
        if event["event"] == "Treatment_Stop":                              # Rule 2: A treatment stop should not precede its start.
            start = dates.get("Treatment_Start")
            if start and event["date"] < start:
                problems.append("Treatment stop precedes treatment start")
        if problems:
            flagged.append((event, problems))
        else:
            accepted.append(event)
    return accepted, flagged

accepted, flagged = validate_events(events)
print("Accepted events:")
for event in accepted:
    print(event["event"], event["date"])
print("\nFlagged events:")
for event, problems in flagged:
    print(event["event"], ":", "; ".join(problems))

Accepted events:
EGFR_Result 2026-01-10
Treatment_Start 2026-01-20

Flagged events:
Treatment_Stop : Treatment stop precedes treatment start
Response_Assessment : Missing evidence source
